# Regressão Linear Múltipla — California Housing

Neste notebook, aplicamos **Regressão Linear Múltipla** ao dataset *California Housing*, mantendo a mesma lógica da aula anterior:

- separação em **treino, validação e teste**
- treinamento no conjunto de **treino**
- análise do desempenho no conjunto de **validação**
- avaliação final no conjunto de **teste**

Agora, em vez de usar apenas uma variável explicativa, utilizaremos **várias variáveis ao mesmo tempo** para prever `MedHouseVal`.

**Etapas**
1. Importar bibliotecas e carregar o dataset
2. Selecionar múltiplas variáveis explicativas
3. Dividir os dados em treino, validação e teste
4. Treinar o modelo de regressão linear múltipla
5. Interpretar os coeficientes
6. Avaliar o modelo com **MSE** e **R²**
7. Comparar com a regressão linear simples
8. Visualizar valores reais vs. previstos e resíduos
9. Realizar tarefas práticas


In [ ]:
# 1) Bibliotecas
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt

# Reprodutibilidade
RANDOM_STATE = 42

## 1. Carregar o dataset

Vamos carregar o *California Housing* via `sklearn`.  
O alvo é `MedHouseVal` (valor mediano das casas em centenas de milhares de dólares).


In [ ]:
housing = fetch_california_housing(as_frame=True)
df = housing.frame.copy()

display(df.head())
print(f"Formato: {df.shape}")

In [ ]:
# Matriz de correlação
correlation_matrix = df.corr()
print(correlation_matrix["MedHouseVal"].sort_values(ascending=False))

## 2. Regressão Linear **Múltipla**: escolha das variáveis explicativas

Na aula anterior, usamos apenas **`MedInc`** como variável explicativa.  
Agora vamos usar um conjunto de variáveis para tentar melhorar a capacidade preditiva do modelo.

Neste notebook, utilizaremos inicialmente:

- `MedInc`
- `HouseAge`
- `AveRooms`
- `Latitude`
- `Longitude`

> Observação: a escolha dessas variáveis busca manter o modelo interpretável e, ao mesmo tempo, mostrar a ideia central da regressão múltipla.


In [ ]:
features = ["MedInc", "HouseAge", "AveRooms", "Latitude", "Longitude"]

X = df[features]
y = df["MedHouseVal"]

display(X.head())
X.describe().T

## 3. Divisão em treino, validação e teste

Vamos manter a mesma estratégia da aula anterior:

- **Treino:** 60%
- **Validação:** 20%
- **Teste:** 20%

Estratégia:
1. Primeiro separamos `train` × `temp` (60/40)
2. Depois dividimos `temp` em `val` × `test` (50/50)


In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.40, random_state=RANDOM_STATE
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=RANDOM_STATE
)

print("Shapes:")
print("  Treino:", X_train.shape)
print("  Validação:", X_val.shape)
print("  Teste:", X_test.shape)

## 4. Treinamento do modelo

Agora ajustaremos um modelo de **Regressão Linear Múltipla**.

**Equação do modelo**:

\[
h(x)=\theta_0 + \theta_1 x_1 + \theta_2 x_2 + \cdots + \theta_n x_n
\]

Mesmo usando várias variáveis, o modelo ainda é **linear**,
pois continua sendo linear nos coeficientes \(\theta\).


In [ ]:
# Criamos o modelo de Regressão Linear Múltipla
model = LinearRegression()

# Ajustamos o modelo usando os dados de treino
model.fit(X_train, y_train)

# Intercepto e coeficientes
theta0 = model.intercept_
coefs = model.coef_

print(f"Intercepto (θ0): {theta0:.6f}")
print("Coeficientes:")
for var, coef in zip(features, coefs):
    print(f"  {var}: {coef:.6f}")

### Interpretação dos coeficientes

Na regressão múltipla, cada coeficiente indica o quanto o valor previsto do target varia
quando a respectiva variável aumenta em 1 unidade, **mantendo as demais fixas**.

Esse detalhe é muito importante:

- em regressão simples, olhamos o efeito de uma única variável;
- em regressão múltipla, olhamos o efeito **parcial** de cada variável no contexto das outras.


In [ ]:
coef_df = pd.DataFrame({
    "Variável": features,
    "Coeficiente": coefs
}).sort_values("Coeficiente", ascending=False)

display(coef_df)

## 5. Avaliação no conjunto de **validação**

Utilizaremos as mesmas métricas da aula anterior:

- **MSE** (Erro Quadrático Médio)
- **R²** (Coeficiente de Determinação)


### Mean Squared Error (Erro Quadrático Médio)

\begin{equation}
\text{MSE} = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2
\end{equation}

Onde:

\begin{equation}
\begin{aligned}
&\bullet y_i \quad\text{= valor real da observação} \\
&\bullet \hat{y}_i \quad\text{= valor previsto pelo modelo} \\
&\bullet n \quad\text{= número total de observações}
\end{aligned}
\end{equation}

Quanto menor o MSE, melhor o modelo está ajustado aos dados.


### R² – Coeficiente de Determinação

\begin{equation}
R^2 = 1 - \frac{\sum_{i=1}^{n} (y_i - \hat{y}_i)^2}{\sum_{i=1}^{n} (y_i - \bar{y})^2}
\end{equation}

Onde:

\begin{equation}
\begin{aligned}
&\bullet \bar{y} \quad\text{= média dos valores reais} \\
&\bullet y_i \quad\text{= valor real da observação} \\
&\bullet \hat{y}_i \quad\text{= valor previsto pelo modelo}
\end{aligned}
\end{equation}

Quanto mais próximo de 1, maior a proporção da variância explicada pelo modelo.


In [ ]:
# Previsões no conjunto de validação
y_val_pred = model.predict(X_val)

# Métricas
mse_val = mean_squared_error(y_val, y_val_pred)
r2_val = r2_score(y_val, y_val_pred)

print(f"MSE (validação): {mse_val:.6f}")
print(f"R²  (validação): {r2_val:.6f}")

### Interpretação dos resultados de validação

Com base nos valores obtidos:

- compare o **MSE** com o notebook de regressão linear simples;
- observe se o **R²** aumentou;
- reflita se o uso de mais variáveis parece ter melhorado a capacidade preditiva do modelo.

> Pergunta: usar várias variáveis melhorou a previsão em relação ao uso de apenas `MedInc`?


## 6. Avaliação final no **teste**

Após analisar o desempenho no conjunto de validação, fazemos a avaliação final no conjunto de teste.
Essa etapa fornece uma estimativa mais confiável da capacidade de **generalização** do modelo.


In [ ]:
y_test_pred = model.predict(X_test)

mse_test = mean_squared_error(y_test, y_test_pred)
r2_test = r2_score(y_test, y_test_pred)

print(f"MSE (teste): {mse_test:.6f}")
print(f"R²  (teste): {r2_test:.6f}")

### Interpretação dos resultados de teste

Analise:

- o desempenho no teste ficou parecido com o da validação?
- houve indícios de boa generalização?
- o modelo múltiplo parece mais adequado que o modelo simples?

Se os valores de validação e teste forem parecidos, isso sugere que o modelo está se comportando de forma estável em novos dados.


## 7. Comparação com a regressão linear simples

No notebook anterior, usando apenas `MedInc`, foram obtidos os seguintes resultados:

- **MSE (validação):** 0.714500
- **R² (validação):** 0.452632
- **MSE (teste):** 0.702809
- **R² (teste):** 0.487422

Agora vamos comparar esses valores com os obtidos na regressão múltipla.


In [ ]:
comparacao = pd.DataFrame({
    "Modelo": ["Linear Simples", "Linear Múltipla"],
    "MSE (validação)": [0.714500, mse_val],
    "R² (validação)": [0.452632, r2_val],
    "MSE (teste)": [0.702809, mse_test],
    "R² (teste)": [0.487422, r2_test],
})

display(comparacao)

## 8. Visualização

Na regressão linear simples era possível visualizar a reta ajustada no plano.

Na regressão múltipla, como temos várias variáveis explicativas, essa visualização direta deixa de ser prática.
Por isso, vamos usar gráficos mais adequados:

1. **Valores reais vs. previstos**
2. **Gráfico de resíduos**


In [ ]:
plt.figure(figsize=(7, 7))
plt.scatter(y_test, y_test_pred, alpha=0.3)
plt.xlabel("Valor real")
plt.ylabel("Valor previsto")
plt.title("Regressão Linear Múltipla — Real vs. Previsto")
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], linewidth=2)
plt.show()

### Comentário sobre o gráfico real vs. previsto

Se o modelo estiver prevendo bem, os pontos tendem a ficar próximos da diagonal.

Quanto mais espalhados estiverem:
- maior tende a ser o erro;
- menor tende a ser a qualidade da previsão.


In [ ]:
residuos = y_test - y_test_pred

plt.figure(figsize=(8, 5))
plt.scatter(y_test_pred, residuos, alpha=0.3)
plt.axhline(0, linestyle="--")
plt.xlabel("Valor previsto")
plt.ylabel("Resíduo")
plt.title("Resíduos do modelo de Regressão Linear Múltipla")
plt.show()

### Comentário sobre os resíduos

Idealmente:

- os resíduos devem estar distribuídos de forma aproximadamente aleatória;
- não deve aparecer um padrão muito forte;
- resíduos muito estruturados podem indicar limitações do modelo.


## 9. Interpretação e comentários

- **Mais variáveis** podem ajudar o modelo a capturar melhor a relação com o target.
- Isso **não garante automaticamente** um modelo melhor, mas frequentemente melhora a previsão.
- A interpretação dos coeficientes exige cuidado, pois o efeito de cada variável é avaliado **mantendo as demais constantes**.
- Este notebook prepara o terreno para os próximos temas:
  - **Polynomial Features**
  - **Regularização**


# Tarefas — Atividade

📌 Este notebook pode ser usado como base para a atividade de regressão linear múltipla.

Cada tarefa deve conter:

- **Código em Python**
- **Markdown explicativo** abaixo do código
- **Comentário interpretando** os resultados


## 1. Experimente outro conjunto de variáveis

Crie um novo modelo usando um conjunto diferente de variáveis explicativas.

Sugestões:
- `["MedInc", "AveOccup", "Latitude", "Longitude"]`
- `["HouseAge", "AveRooms", "AveBedrms", "Population"]`

Perguntas:
- O desempenho melhorou ou piorou?
- Quais variáveis parecem mais úteis?


In [ ]:
# Monte aqui um novo conjunto de variáveis
novas_features = []

# Exemplo:
# novas_features = ["MedInc", "AveOccup", "Latitude", "Longitude"]

# Seu código aqui


## 2. Analise os coeficientes do novo modelo

Depois de treinar o novo modelo:

- liste os coeficientes;
- identifique coeficientes positivos e negativos;
- tente interpretar o significado deles.


In [ ]:
# Treine aqui o novo modelo e exiba os coeficientes


## 3. Compare com a regressão linear simples

Use os resultados do notebook anterior e compare:

- MSE
- R²
- capacidade de generalização

Pergunta:
> Valeu a pena passar de regressão simples para regressão múltipla?


## 4. Reflexão crítica

Responda:

- Por que várias variáveis podem melhorar a previsão?
- Ainda existe alguma limitação em usar apenas regressão linear múltipla?
- Que tipo de relação esse modelo pode não capturar bem?


## Reflexão final

A regressão linear múltipla é uma evolução natural da regressão linear simples:

- antes: **uma variável explicativa**
- agora: **várias variáveis explicativas**

Mesmo assim, o modelo continua sendo **linear**, pois continua linear nos coeficientes.

No próximo passo, podemos explorar:
- **Polynomial Features** para capturar curvaturas;
- **Regularização** para controlar a complexidade do modelo.
